# DaT Parkinson's: an ordered pipeline walkthrough

This notebook explains the package without downloading patient data or pretrained weights.
Run `python -m pip install -e ".[dev]"` from the repository root in your environment first.
The original experiments are documented in `docs/source-map.md`; the values below are historical, not reproduced results.


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import display
from PIL import Image
from dat_parkinsons.demo import synthetic_volume
from dat_parkinsons.preprocessing import ViewConfig, make_view, raw_scorer_crop, scorer_input

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the repository"

## 1. Select a slice

The trained selector scores each axial array slice. Its input is the rotated central 50% crop, windowed at percentiles 50 and 99, resized to 128 x 128, with one channel. Group validation slices by patient.

Here we choose z=8 only to illustrate preprocessing on a synthetic volume. This is not a trained selection result.


In [ ]:
volume = synthetic_volume(seed=42)
z = 8
selector_tensor = scorer_input(raw_scorer_crop(volume, z))
print("Volume:", volume.shape, "Selector input:", selector_tensor.shape)
display(Image.fromarray((selector_tensor[0] * 255).astype(np.uint8)).resize((256, 256)))

## 2. Build complementary views

RGB combines percentile intensity, CLAHE, and an uptake ratio map. The 2.5D representations use three slabs around the selected slice. Each slab averages the center slice and its available immediate neighbors. Normalization is within one examination. Values are quantized exactly as in the PNG training path.


In [ ]:
configs = [
    ViewConfig(mode="rgb"),
    ViewConfig(offset_mm=2),
    ViewConfig(offset_mm=4),
    ViewConfig(offset_mm=7),
]
for config in configs:
    image, uptake_features = make_view(volume, z, config)
    print(config.mode, "offset", config.offset_mm, "uptake proxies", uptake_features)
    display(Image.fromarray(image))

## 3. Train classifiers with explicit configurations

Run the CLI commands in the README after obtaining appropriate data rights. Each classifier run saves its folds, training medians, preprocessing, scorer hash, and checkpoint metadata. Shared patient folds avoid hidden differences between branch experiments.

The optional uptake features are geometric proxies, not anatomically segmented clinical measurements.


In [ ]:
config = json.loads((ROOT / "configs/swin_z7.json").read_text())
print(json.dumps(config, indent=2))

## 4. Blend in logit space

Weights are nonnegative and sum to one. This synthetic example demonstrates probability arithmetic only. Fit weights on training-side predictions, then evaluate on a disjoint cohort that was also excluded from upstream training and selection.


In [ ]:
from dat_parkinsons.ensemble import blend

example = np.array([[0.2, 0.8], [0.1, 0.3]])
print(blend(example, [0.5, 0.5]))

## 5. Interpret historical results carefully

The five-model blend reports 0.2189 log loss / 0.9693 AUROC on its weight-fitting data. It is not a private leaderboard or independent test result. Duplicate provenance, selector training overlap, and checkpoint selection need to be resolved before strong generalization claims.


In [ ]:
results = json.loads((ROOT / "reports/historical-results.json").read_text())
print(json.dumps(results, indent=2))

## 6. Verify the complete software path

Run `dat-parkinsons demo --output runs/demo` from a terminal. This creates synthetic NIfTI input and random checkpoints, loads an inference bundle, writes `submission.csv`, and checks repeatability. No accuracy claim follows from this test.

Next reading: `docs/reproduction.md`, `docs/evaluation.md`, and `docs/model-card.md`.
